# Aspirantes Cup: modelado y optimización

Flujo: cargar datos, explorar estrategias de neumáticos, medir el efecto de cada atributo, entrenar un modelo sustituto y buscar el mejor reparto de puntos.

Las secciones 4 y 5 se activan solas cuando hay datos suficientes (experimentos en la Academia y 10 o más carreras con resultado).

In [ ]:
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

D = Path("data")
ATTRS = ["speed","acceleration","aero","handling","braking",
         "reliability","tyre_management","wet_weather","overtaking","consistency"]
W_COLS = ["tyre_wear","w_speed","w_accel","w_aero","w_handling","w_braking"]

races = pd.read_csv(D/"races.csv")
stats = pd.read_csv(D/"car_stats.csv")
strat = pd.read_csv(D/"strategy.csv")
tyres = pd.read_csv(D/"tyres.csv")
exps  = pd.read_csv(D/"experiments.csv")
df = races.merge(stats, on="race_no").merge(strat, on="race_no")
done = df.dropna(subset=["position"])
print(f"Carreras: {len(df)} | con resultado: {len(done)} | experimentos: {len(exps)}")
df

## 1. Ficha de la próxima carrera
Codificación: 1 = 🔼, 0 = neutro, -1 = 🔽.

In [ ]:
race = df.iloc[-1]
print(race[["race_no","track","laps","rain"] + W_COLS].to_string())

## 2. Estrategias de neumáticos viables
`wear_mult` y `style_mult` se calibran con `/practice tyres`. Por ahora valen 1.0 (desgaste medio, balanced).

In [ ]:
LIFE = dict(zip(tyres.code, tyres.base_life_laps))
DRY = ["S","M","H"]

def candidates(laps, wear_mult=1.0, style_mult=1.0, max_stops=2):
    rows = []
    for n in range(1, max_stops+2):
        for combo in itertools.product(DRY, repeat=n):
            stint = laps/n
            margin = min(LIFE[c]*wear_mult*style_mult - stint for c in combo)
            rows.append({"strategy":"-".join(combo), "stops":n-1,
                         "viable":margin>=0, "min_margin_laps":round(margin,1)})
    return pd.DataFrame(rows).sort_values(["viable","min_margin_laps"], ascending=False)

cand = candidates(int(race.laps), wear_mult=1.0, style_mult=1.0)
cand.head(15)

In [ ]:
top = cand[cand.stops<=1].drop_duplicates("strategy")
colors = np.where(top.viable, "tab:green", "tab:red")
plt.figure(figsize=(8,4))
plt.barh(top.strategy, top.min_margin_laps, color=colors)
plt.axvline(0, color="k", lw=.8)
plt.xlabel("Margen mínimo de desgaste (vueltas)")
plt.title(f"Estrategias de 0 y 1 parada, {race.track} ({int(race.laps)} vueltas)")
plt.tight_layout(); plt.show()

## 3. Calibración de desgaste
Cuando tengas la salida de `/practice tyres`, anota la vida real observada y calcula el multiplicador.
Ejemplo: si el soft aguanta 18 vueltas en esta pista, `wear_mult = 18/15 = 1.2`.

In [ ]:
def wear_multiplier(compound, observed_laps):
    return observed_laps / LIFE[compound]

# wear_multiplier("S", 18)

## 4. Efecto de cada atributo (experimentos en la Academia)
Rellena `data/experiments.csv`: sube un atributo, repite `/practice pace` y anota la métrica (ritmo, tiempo de vuelta, etc.).

In [ ]:
if len(exps):
    g = exps.groupby("attribute").apply(
        lambda t: np.polyfit(t.points_added, t.metric_value, 1)[0] if t.points_added.nunique()>1 else np.nan
    ).rename("efecto_por_punto").sort_values()
    display(g)
    g.plot.barh(title="Efecto marginal por punto (según metric_name)")
    plt.tight_layout(); plt.show()
else:
    print("Sin experimentos todavía. Añade filas a data/experiments.csv.")

## 5. Modelo sustituto con resultados reales
Entrena una vez que haya 10 o más carreras con posición.

In [ ]:
model = None
if len(done) >= 10:
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.model_selection import cross_val_score
    from sklearn.inspection import permutation_importance
    FEATS = ATTRS + ["laps"] + W_COLS
    X, y = done[FEATS], done["position"]
    model = GradientBoostingRegressor(random_state=0)
    print("MAE CV (posiciones):", -cross_val_score(model, X, y, cv=min(5,len(done)), scoring="neg_mean_absolute_error").mean())
    model.fit(X, y)
    imp = permutation_importance(model, X, y, n_repeats=20, random_state=0)
    pd.Series(imp.importances_mean, index=FEATS).sort_values().plot.barh(title="Importancia de variables")
    plt.tight_layout(); plt.show()
else:
    print(f"Faltan {10-len(done)} carreras con resultado para entrenar.")

## 6. Optimización del reparto de puntos
Búsqueda aleatoria sobre repartos que suman el presupuesto, evaluados con el modelo sustituto.

In [ ]:
def best_allocation(model, base_stats, race_row, budget, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    best, best_pred = None, np.inf
    for _ in range(n):
        alloc = np.floor(rng.dirichlet(np.ones(len(ATTRS)))*budget).astype(int)
        row = {a: base_stats[a]+k for a,k in zip(ATTRS, alloc)}
        row.update({"laps":race_row.laps, **{c:race_row[c] for c in W_COLS}})
        pred = model.predict(pd.DataFrame([row])[ATTRS+["laps"]+W_COLS])[0]
        if pred < best_pred:
            best, best_pred = dict(zip(ATTRS, alloc)), pred
    return best, best_pred

if model is not None:
    base = stats.iloc[-1]
    alloc, pred = best_allocation(model, base, race, budget=int(base.upgrade_points_available))
    print("Posición esperada:", round(pred,2)); print(alloc)
else:
    print("Se activa cuando el modelo esté entrenado.")